PROJETO: Análise de Vendas Automotivas<br>
ETAPA: Extração de Dados<br>
ORIGEM: PostgreSQL<br>
DESTINO INICIAL: Dataset/Brutos

#### Instalando as dependências

In [ ]:

pip install pandas sqlalchemy psycopg2-binary python-dotenv google-cloud-bigquery

#### Importando as bibliotecas

In [1]:
import pandas as pd
import os

from dotenv import load_dotenv
from sqlalchemy import create_engine, URL, text

#### Carregando variáveis de conexão do arquivo .env

In [2]:
load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT"))
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

#### Conexão com PostgreSQL

In [3]:
connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

engine = create_engine(connection_url)

#### Teste de conexão

In [4]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT version(), current_database()")
    )

    print(result.fetchone())

('PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35229, 64-bit', 'postgres')


#### Extração dos dados

In [5]:
# Lista das tabelas que serão extraídas do PostgreSQL

tables = {
    "customers": "sales",
    "funnel": "sales",
    "products": "sales",
    "stores": "sales",
    "ibge_genders": "temp_tables",
    "regions": "temp_tables"
}

In [6]:
# Dicionário que armazena os DataFrames

dataframes = {}

for table, schema in tables.items():

    query = f"""
        SELECT *
        FROM {schema}.{table}
    """

    dataframes[table] = pd.read_sql_query(
        query,
        engine
    )

In [ ]:
# Atribuindo os DataFrames correspondentes às tabelas extraídas do banco de dados.
df_customers = dataframes["customers"]
df_funnel = dataframes["funnel"]
df_products = dataframes["products"]
df_stores = dataframes["stores"]
df_ibge_genders = dataframes["ibge_genders"]
df_regions = dataframes["regions"]

In [ ]:
# Resumo da extração
resumo = pd.DataFrame([
    {
        "tabela": tabela,
        "registros": df.shape[0],
        "colunas": df.shape[1]
    }
    for tabela, df in dataframes.items()
])

resumo

,tabela,registros,colunas
0,customers,25109,12
1,funnel,30580,10
2,products,333,5
3,stores,3026,3
4,ibge_genders,100787,2
5,regions,5570,6


In [ ]:
import os

# Diretório de armazenamento
os.makedirs("../Dataset/csv", exist_ok=True)

# Exportação dos DataFrames
for tabela, df in dataframes.items():

    df.to_csv(
        f"../Dataset/csv/{tabela}.csv",
        index=False,
        encoding="utf-8"
    )

    print(f"{tabela}.csv exportado: {len(df)} registros")

customers.csv exportado: 25109 registros
funnel.csv exportado: 30580 registros
products.csv exportado: 333 registros
stores.csv exportado: 3026 registros
ibge_genders.csv exportado: 100787 registros
regions.csv exportado: 5570 registros
